#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [2]:
import json
import sys
from datetime import datetime, date, timedelta
from typing import Dict, List, Any, Optional, Set


from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, udf, lit, when, isnan, isnull, to_json, from_json,
    explode, collect_list, struct, array, size, coalesce,
    to_timestamp, datediff, current_date, regexp_extract, broadcast, get_json_object
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, BooleanType, MapType, ArrayType, DoubleType
)
from pyspark.sql import functions as F
from itertools import product

import logging

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# 常量定义
ACCOUNT_TYPES = ['conta_deposito_a_vista', 'conta_poupanca', 'conta_pagamento_pre_paga']
ACCOUNT_SUBTYPES = ['individual', 'conjunta_simples', 'conjunta_solidaria']

def generate_all_feature_names() -> Set[str]:
    """生成所有可能的特征名称"""
    features = set()
    
    # Account Base Info - Indicator
    features.add('inst_acct_has_data')
    features.add('inst_acct_days_from_last_update')
    
    # Account Base Info - Account
    features.add('inst_acct_cnt')
    
    # Account Type Features
    for acct_type in ACCOUNT_TYPES:
        features.add(f'inst_acct_{acct_type}_cnt')
        features.add(f'inst_acct_{acct_type}_cnt_ratio')
    
    # Account Subtype Features
    for acct_subtype in ACCOUNT_SUBTYPES:
        features.add(f'inst_acct_{acct_subtype}_cnt')
        features.add(f'inst_acct_{acct_subtype}_cnt_ratio')
    
    # Balance - Indicator
    features.add('inst_acct_bal_has_data')
    features.add('inst_acct_bal_days_from_last_update')
    
    # Balance - Total Balance
    features.add('inst_acct_bal_total_amt')
    for acct_type in ACCOUNT_TYPES:
        features.add(f'inst_acct_bal_{acct_type}_total_amt')
        features.add(f'inst_acct_bal_{acct_type}_total_amt_ratio')
    
    # Balance - Available Balance
    features.add('inst_acct_bal_total_available_amt')
    features.add('inst_acct_bal_total_available_amt_ratio')
    for acct_type in ACCOUNT_TYPES:
        features.add(f'inst_acct_bal_{acct_type}_total_available_amt')
        features.add(f'inst_acct_bal_{acct_type}_total_available_amt_ratio')
    
    # Balance - Blocked Balance
    features.add('inst_acct_bal_total_blocked_amt')
    features.add('inst_acct_bal_total_blocked_amt_ratio')
    for acct_type in ACCOUNT_TYPES:
        features.add(f'inst_acct_bal_{acct_type}_total_blocked_amt')
        features.add(f'inst_acct_bal_{acct_type}_total_blocked_amt_ratio')
    
    # Balance - Auto Investment Balance
    features.add('inst_acct_bal_total_auto_invest_amt')
    features.add('inst_acct_bal_total_auto_invest_amt_ratio')
    for acct_type in ACCOUNT_TYPES:
        features.add(f'inst_acct_bal_{acct_type}_total_auto_invest_amt')
        features.add(f'inst_acct_bal_{acct_type}_total_auto_invest_amt_ratio')
    
    return features

def extract_account_number_from_url(url: str) -> Optional[str]:
    """从URL中提取账户ID"""
    parts = url.split("/")
    # 寻找最后一个"accounts"后面的账户ID
    for i in reversed(range(len(parts))):
        if parts[i] == "accounts" and i + 1 < len(parts):
            account_id = parts[i + 1].split("?")[0]  # 移除查询参数
            # 确保这不是版本号或其他关键字
            if account_id not in ['v1', 'v2', 'v3']:
                return account_id
    return None

def calculate_all_features(all_accounts: List[Dict], current_date: datetime, latest_update_time: Optional[datetime]) -> Dict[str, Any]:
    """计算所有特征"""
    features = {}

    # Account Base Info - Indicator
    features['inst_acct_has_data'] = 1.0 if all_accounts else 0.0
    features['inst_acct_days_from_last_update'] = (
        round((current_date - latest_update_time).total_seconds() / 86400)
        if latest_update_time else -9999
    )

    # Account Base Info - Account
    total_accounts = len(all_accounts)
    features['inst_acct_cnt'] = total_accounts

    # Account Type Features
    accounts_by_type = {}
    for account in all_accounts:
        acct_type = account['type'].lower() if account['type'] else 'unknown'
        if acct_type not in accounts_by_type:
            accounts_by_type[acct_type] = []
        accounts_by_type[acct_type].append(account)

    for acct_type in ACCOUNT_TYPES:
        type_count = len(accounts_by_type.get(acct_type, []))
        features[f'inst_acct_{acct_type}_cnt'] = type_count
        features[f'inst_acct_{acct_type}_cnt_ratio'] = type_count / total_accounts if total_accounts > 0 else 0.0

    # Account Subtype Features
    accounts_by_subtype = {}
    for account in all_accounts:
        acct_subtype = account['subtype'].lower() if account['subtype'] else 'unknown'
        if acct_subtype not in accounts_by_subtype:
            accounts_by_subtype[acct_subtype] = []
        accounts_by_subtype[acct_subtype].append(account)

    for acct_subtype in ACCOUNT_SUBTYPES:
        subtype_count = len(accounts_by_subtype.get(acct_subtype, []))
        features[f'inst_acct_{acct_subtype}_cnt'] = subtype_count
        features[f'inst_acct_{acct_subtype}_cnt_ratio'] = subtype_count / total_accounts if total_accounts > 0 else 0.0

    # Balance Features
    has_balance_data = any(account['balances']['available'] > 0 or 
                          account['balances']['blocked'] > 0 or 
                          account['balances']['autoInvested'] > 0 
                          for account in all_accounts)
    features['inst_acct_bal_has_data'] = 1.0 if has_balance_data else 0.0

    latest_balance_update = max((account['updateDateTime'] for account in all_accounts if account['updateDateTime']), default=None)
    features['inst_acct_bal_days_from_last_update'] = (
        round((current_date - latest_balance_update).total_seconds() / 86400)
        if latest_balance_update else -9999
    )

    # Calculate total amounts
    total_balance = sum(
        account['balances']['available'] + 
        account['balances']['blocked'] + 
        account['balances']['autoInvested']
        for account in all_accounts
    )
    features['inst_acct_bal_total_amt'] = total_balance

    # Calculate type-specific balances
    for acct_type in ACCOUNT_TYPES:
        type_accounts = accounts_by_type.get(acct_type, [])
        type_total = sum(
            account['balances']['available'] + 
            account['balances']['blocked'] + 
            account['balances']['autoInvested']
            for account in type_accounts
        )
        features[f'inst_acct_bal_{acct_type}_total_amt'] = type_total
        features[f'inst_acct_bal_{acct_type}_total_amt_ratio'] = type_total / total_balance if total_balance > 0 else 0.0

    # Available Balance Features
    total_available = sum(account['balances']['available'] for account in all_accounts)
    features['inst_acct_bal_total_available_amt'] = total_available
    features['inst_acct_bal_total_available_amt_ratio'] = total_available / total_balance if total_balance > 0 else 0.0

    for acct_type in ACCOUNT_TYPES:
        type_accounts = accounts_by_type.get(acct_type, [])
        type_available = sum(account['balances']['available'] for account in type_accounts)
        features[f'inst_acct_bal_{acct_type}_total_available_amt'] = type_available
        features[f'inst_acct_bal_{acct_type}_total_available_amt_ratio'] = type_available / total_available if total_available > 0 else 0.0

    # Blocked Balance Features
    total_blocked = sum(account['balances']['blocked'] for account in all_accounts)
    features['inst_acct_bal_total_blocked_amt'] = total_blocked
    features['inst_acct_bal_total_blocked_amt_ratio'] = total_blocked / total_balance if total_balance > 0 else 0.0

    for acct_type in ACCOUNT_TYPES:
        type_accounts = accounts_by_type.get(acct_type, [])
        type_blocked = sum(account['balances']['blocked'] for account in type_accounts)
        features[f'inst_acct_bal_{acct_type}_total_blocked_amt'] = type_blocked
        features[f'inst_acct_bal_{acct_type}_total_blocked_amt_ratio'] = type_blocked / total_blocked if total_blocked > 0 else 0.0

    # Auto Investment Balance Features
    total_auto_invest = sum(account['balances']['autoInvested'] for account in all_accounts)
    features['inst_acct_bal_total_auto_invest_amt'] = total_auto_invest
    features['inst_acct_bal_total_auto_invest_amt_ratio'] = total_auto_invest / total_balance if total_balance > 0 else 0.0

    for acct_type in ACCOUNT_TYPES:
        type_accounts = accounts_by_type.get(acct_type, [])
        type_auto_invest = sum(account['balances']['autoInvested'] for account in type_accounts)
        features[f'inst_acct_bal_{acct_type}_total_auto_invest_amt'] = type_auto_invest
        features[f'inst_acct_bal_{acct_type}_total_auto_invest_amt_ratio'] = type_auto_invest / total_auto_invest if total_auto_invest > 0 else 0.0

    return features

def get_cpf_account_profile(cpf: str, get_open_account_data: str, feature_list: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    获取CPF账户档案

    Args:
        cpf: CPF号码
        get_open_account_data: 开放账户数据JSON字符串
        feature_list: 可选的特征列表，如果提供则只返回指定的特征

    Returns:
        包含特征组的字典
    """
    parsed_data = json.loads(get_open_account_data)
    # 检查数据格式并适配
    if isinstance(parsed_data, list):
        # 新格式：直接是机构列表
        open_account_data = parsed_data
    elif isinstance(parsed_data, dict) and 'data' in parsed_data:
        # 旧格式：包含在data字段中
        open_account_data = parsed_data['data']
    elif isinstance(parsed_data, dict) and 'get_open_finance_personal_account_data' in parsed_data:
        # DataFrame格式：包含cpf和get_open_finance_personal_account_data字段
        account_data = parsed_data['get_open_finance_personal_account_data']
        if isinstance(account_data, str):
            # 如果是字符串，需要再次解析
            try:
                open_account_data = json.loads(account_data)
            except:
                open_account_data = []
        elif isinstance(account_data, list):
            # 如果已经是列表，直接使用
            open_account_data = account_data
        else:
            open_account_data = []
    else:
        # 未知格式，尝试直接使用
        open_account_data = parsed_data if isinstance(parsed_data, list) else []

    if not open_account_data:
        return {"cpf": cpf, "feature_group": {name: 0.0 for name in generate_all_feature_names()}}

    # 当前日期
    current_date = datetime.now()

    # 存储所有账户信息
    all_accounts = []
    latest_update_time = None

    # 处理开放账户数据
    for i, institution in enumerate(open_account_data):
        # 适配不同的字段名称
        institution_name = institution.get('organisation_name') or institution.get('机构_name', 'unknown')
        institution_data = institution.get('data', [])
        
        
        for j, api_call in enumerate(institution_data):
            url = api_call['url']
            # 处理rsp字段，可能是字符串需要解析
            rsp = api_call['rsp']
            if isinstance(rsp, str):
                try:
                    rsp = json.loads(rsp)
                    # 如果解析后有data字段，使用data字段的内容
                    if isinstance(rsp, dict) and 'data' in rsp:
                        rsp = rsp['data']
                except (json.JSONDecodeError, TypeError):
                    # 如果解析失败，使用原始数据
                    continue

            # 判断URL类型
            def is_accounts_list_url(url):
                """判断是否为账户列表API"""
                # 去掉查询参数
                base_url = url.split('?')[0]
                # 检查是否以 /v2/accounts 结尾 (没有具体的account_id)
                return base_url.endswith('/v2/accounts')
            
            def is_account_detail_url(url):
                """判断是否为账户详情API"""
                # 去掉查询参数
                base_url = url.split('?')[0]
                # 只查找 /v2/accounts/ 后面的部分（确保找的是最后一个accounts）
                if '/v2/accounts/' in base_url:
                    after_pattern = base_url.split('/v2/accounts/')[-1]
                    # 账户详情URL应该只有account_id，没有更多子路径
                    return after_pattern and '/' not in after_pattern
                return False
            
            def is_account_sub_resource_url(url):
                """判断是否为账户子资源API"""
                # 去掉查询参数
                base_url = url.split('?')[0]
                # 只查找 /v2/accounts/ 后面的部分（确保找的是最后一个accounts）
                if '/v2/accounts/' in base_url:
                    after_pattern = base_url.split('/v2/accounts/')[-1]
                    # 子资源URL应该有account_id和子资源路径
                    return after_pattern and '/' in after_pattern
                return False

            # 处理账户列表API - accounts/v2/accounts
            if is_accounts_list_url(url):
                if isinstance(rsp, list):
                    for k, account in enumerate(rsp):
                        account_info = {
                            'type': account.get('type'),
                            'subtype': None,  # 将在账户详情中更新
                            'number': account.get('number'),
                            'accountId': account.get('accountId'),  # 添加accountId
                            'balances': {
                                'available': 0.0,
                                'blocked': 0.0,
                                'autoInvested': 0.0
                            },
                            'updateDateTime': None
                        }
                        all_accounts.append(account_info)
            # 处理账户详情API - accounts/v2/accounts/{account_id}
            elif is_account_detail_url(url):
                account_number = rsp.get('number')
                for account in all_accounts:
                    if account['number'] == account_number:
                        account['subtype'] = rsp.get('subtype')

            # 处理账户子资源API - accounts/v2/accounts/{account_id}/balances 等
            elif is_account_sub_resource_url(url):
                account_id = extract_account_number_from_url(url)
                
                # 根据子资源类型处理不同的数据
                if "/balances" in url:
                    # 处理余额信息
                    # 处理更新时间，支持多种格式
                    update_time = None
                    if isinstance(rsp, dict) and 'updateDateTime' in rsp:
                        try:
                            update_time_str = rsp['updateDateTime']
                            if update_time_str:
                                # 尝试不同的时间格式
                                for fmt in ['%Y-%m-%dT%H:%M:%SZ', '%Y-%m-%dT%H:%M:%S.%fZ']:
                                    try:
                                        update_time = datetime.strptime(update_time_str, fmt)
                                        break
                                    except ValueError:
                                        continue
                        except (ValueError, TypeError):
                            update_time = None
                    
                    if update_time and (latest_update_time is None or update_time > latest_update_time):
                        latest_update_time = update_time

                    matched_balance = False
                    for account in all_accounts:
                        # 使用accountId进行匹配，如果没有accountId则使用number
                        if account.get('accountId') == account_id or account.get('number') == account_id:
                            if isinstance(rsp, dict):
                                available = float(rsp.get('availableAmount', {}).get('amount', 0) or 0)
                                blocked = float(rsp.get('blockedAmount', {}).get('amount', 0) or 0)
                                auto_invested = float(rsp.get('automaticallyInvestedAmount', {}).get('amount', 0) or 0)
                                
                                account['balances']['available'] = available
                                account['balances']['blocked'] = blocked
                                account['balances']['autoInvested'] = auto_invested
                                account['updateDateTime'] = update_time
                                
                                matched_balance = True
                                break
                
                # 可以在这里添加其他子资源的处理逻辑
                # elif "/transactions" in url:
                #     # 处理交易信息
                # elif "/overdraft-limits" in url:
                #     # 处理透支限制信息


    features = calculate_all_features(all_accounts, current_date, latest_update_time)

    # 显示关键特征
    key_features = ['inst_acct_has_data', 'inst_acct_cnt', 'inst_acct_days_from_last_update', 
                   'inst_acct_bal_has_data', 'inst_acct_bal_total_amt']
    for feature in key_features:
        value = features.get(feature, 'NOT_FOUND')

    # 获取所有可能的特征名称
    all_feature_names = generate_all_feature_names()
    
    # 如果没有提供feature_list，使用所有可能的特征
    if feature_list is None:
        feature_list = list(all_feature_names)
    
    # 构建最终的特征字典
    filtered_features = {}
    for feature_name in feature_list:
        if feature_name in features:
            filtered_features[feature_name] = features[feature_name]
        else:
            # 如果特征不存在，设置默认值
            filtered_features[feature_name] = 0.0
    features = filtered_features

    return {
        "cpf": cpf,
        "feature_group": features
    }

def read_source_data(ds, column, source_table="risk.dws_risk_br_openfinance_df"):
    """
    从源表读取数据
    Args:
        source_table: 源表名称
    Returns:
        DataFrame包含所需的列
    """

    df = spark.table(source_table)
    # 选择所需的列
    selected_df = df.select(
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        col(f"{column}"),
    ).filter(
        col("get_open_finance_personal_account_data").isNotNull() &
        (col("get_open_finance_personal_account_data") != "") &
        (col("grass_date") == ds) 
    )

    return selected_df

def write_to_hive(spark_df, category='', schema="dev_risk", table="ads_risk_br_openfinance_df", env="",
                  ds="2025-09-17"):
    """
    将 Spark DataFrame 写入 Hive 表 (支持分区覆盖)
    """

    root_path = f"/projects/risk/hdfs/{env}/risk_mart/ads/ads_risk_openfinance_df/grass_region=BR"
    # 添加分区字段 grass_date
    spark_df_with_partition = spark_df \
        .withColumn("grass_date", F.lit(ds)) \
        .withColumn("category", F.lit(category))

    # 覆盖写入 Hive (支持分区)
    (
        spark_df_with_partition
        .write.mode("overwrite")
        .format("parquet")
        .partitionBy("category", "grass_date")
        .option("partitionOverwriteMode", "dynamic")
        .option("parquet.compression", "snappy")  # 避免 zstd 兼容问题
        .option("spark.sql.parquet.writeLegacyFormat", "true")  # 避免 decimal/时间戳兼容
        .parquet(f"{root_path}")
    )

    print(f"✅ Data written to Hive table: {schema}.{table} partition grass_date={ds}")
    print(f"📊 Total records written: {spark_df_with_partition.count()}")

    # 如果 Hive Metastore 需要同步 HDFS 分区，可以在 Spark SQL 里调用 MSCK
    spark.sql(f"MSCK REPAIR TABLE {schema}.{table}")


def process_row(raw_data, cpf):
        
    result = get_cpf_account_profile(cpf, raw_data)
    # 确保返回值格式正确
    feature_group = result.get("feature_group", {})
    # 将所有值转换为字符串以符合MapType定义
    feature_group_str = {k: f"{v}" if v is not None else None for k, v in feature_group.items()}

    feature_groups = {}
    feature_groups.update(feature_group_str)

    return {
        "feature_group": feature_groups
    }


def process_dataframe(input_df):
    """
    处理输入DataFrame

    Args:
        input_df: 输入DataFrame，包含shopee_user_id, cpf, get_open_finance_personal_account_data

    Returns:
        处理后的DataFrame，包含shopee_user_id, cpf, origin_data, feature_group
    """
    logger.info("开始处理DataFrame...")

    # 检查输入数据
    input_count = input_df.count()
    logger.info(f"输入数据记录数: {input_count}")

    # 缓存输入DataFrame以提高性能
    input_df.cache()

    # 定义返回类型
    return_schema = StructType([
        StructField("feature_group", MapType(StringType(), StringType(), True), True)
    ])
    # 创建处理UDF
    process_udf = udf(process_row, return_schema)


    # 应用UDF处理数据，添加重分区以优化性能
    result_df = input_df.repartition(200).withColumn(
        "processed_result",
        process_udf(
            col("get_open_finance_personal_account_data"),
            col("cpf")
        )
    )


    # 展开处理结果
    final_df = result_df.select(
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        to_json(col("processed_result.feature_group")).alias("feature_group"),
        col("get_open_finance_personal_account_data").alias("origin_data"),

    )

    # 缓存结果DataFrame
    final_df.cache()

    # 检查输出数据质量
    output_count = final_df.count()
    logger.info(f"输出数据记录数: {output_count}")

    if output_count != input_count:
        logger.warning(f"输入输出记录数不匹配！输入: {input_count}, 输出: {output_count}")

    # 检查空值情况
    null_cpf_count = final_df.filter(col("cpf").isNull() | (col("cpf") == "")).count()
    null_feature_count = final_df.filter(col("feature_group").isNull()).count()

    logger.info(
        f"数据质量检查 - CPF空值: {null_cpf_count}, feature_group: {null_feature_count}")

    return final_df

In [3]:
df = read_source_data(ds='${ds}', column="get_open_finance_personal_account_data")
final_df = process_dataframe(df)
write_to_hive(final_df, category='account_profile', schema='${SCHEMA}', table="ads_risk_br_openfinance_df",
                  env='${ENV}', ds='${ds}')

✅ Data written to Hive table: dev_risk.ads_risk_br_openfinance_df partition grass_date=2025-09-28
📊 Total records written: 157225